In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import re, warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR

LEVELS_FILE  = "Code Outputs/Gap Interpolation Outputs/Unified_Interpolated_Levels.xlsx"
CLIMATE_FILE = "Code Outputs/Climate Data Extraction Outputs/Lake_Climate_Monthly.xlsx"
ORDERS_FILE  = "Code Outputs/Arima Forecast Outputs/FC_orders.csv"
DMI_FILE     = "Climate Indices/dmi.csv"
DIAG_DIR = "Code Outputs/VARX Diagnostic Outputs"; os.makedirs(DIAG_DIR, exist_ok=True)
START, END   = "1995-06-01", "2025-12-01"
TEST_MONTHS  = 60
HORIZONS     = [1, 3, 6, 12]
VAR_MAXLAG   = 4

def rmse(p, a): p, a = np.asarray(p, float), np.asarray(a, float); return np.sqrt(np.nanmean((p - a) ** 2))
def mae(p, a): p, a = np.asarray(p, float), np.asarray(a, float); return np.nanmean(np.abs(p - a))    

# Load everything
lev = pd.read_excel(LEVELS_FILE); lev["Date"] = pd.to_datetime(lev["Date"])
L = (lev.pivot(index="Date", columns="Reservoir", values="Level_m")
        .sort_index().asfreq("MS").loc[START:END])
assert L.notna().all().all(), "NaNs in common window"
LAKES = list(L.columns); dL = L.diff()

clim = pd.read_excel(CLIMATE_FILE); clim["Date"] = pd.to_datetime(clim["Date"])
precip = clim.pivot(index="Date", columns="Reservoir", values="precip_mm").asfreq("MS")

SPLIT_CLIM = len(L) - TEST_MONTHS          # place after L and TEST_MONTHS exist

def deseason(s):                            # training-only climatology
    base = s.iloc[:SPLIT_CLIM]
    monthly = base.groupby(base.index.month).mean()
    return s - s.index.month.map(monthly).to_numpy()

precip_anom = precip.reindex(L.index).apply(deseason)   # reindex before deseasoning

def parse_index(path):
    if not os.path.exists(path): return None
    recs = []
    for line in open(path):
        t = re.split(r"[,\s]+", line.strip())
        if not t or t == [""]: continue
        m = re.match(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})$", t[0])
        if m and len(t) >= 2:
            try: recs.append((pd.Timestamp(int(m[1]), int(m[2]), 1), float(t[1])))
            except ValueError: pass
        elif len(t) == 13 and re.fullmatch(r"\d{4}", t[0]):
            try:
                for mo, v in enumerate([float(x) for x in t[1:]], 1):
                    recs.append((pd.Timestamp(int(t[0]), mo, 1), v))
            except ValueError: pass
    if not recs: return None
    s = pd.Series(dict(recs)).sort_index(); s[s < -90] = np.nan
    return s.asfreq("MS")

dmi = parse_index(DMI_FILE)
dmi = (dmi.reindex(L.index) if dmi is not None else pd.Series(0.0, index=L.index)).ffill().fillna(0.0)

orders = pd.read_csv(ORDERS_FILE).set_index("Lake")

# exogenous blocks
SEAS = pd.get_dummies(L.index.month, prefix="m", drop_first=True).astype(float); SEAS.index = L.index
PRECIP_ALL = precip_anom.add_prefix("precip_")           # all 7 lakes
N = len(L); split = N - TEST_MONTHS; targets = range(split, N)
CLIM_COLS_ALL = list(PRECIP_ALL.columns) + ["DMI"]


# load the shared baseline instead of re-fitting SARIMA
BASE_PRED = "Code Outputs/Baseline Outputs/Baseline_predictions.csv"
bp = pd.read_csv(BASE_PRED, parse_dates=["Origin_Date", "Target_Date"])

assert set(bp["Lake"]) == set(LAKES), "baseline lakes differ from this script's"
idx_of = {d: i for i, d in enumerate(L.index)}

preds, loaded = {}, 0
for r in bp.itertuples(index=False):
    if r.Model not in ("RandomWalk", "SARIMA"):
        continue
    t = idx_of.get(r.Target_Date)
    if t is None:
        continue                      # baseline target outside this window
    preds[(r.Model, r.Lake, t, r.Horizon_m)] = r.Pred_m
    loaded += 1
print(f"  loaded {loaded} shared-baseline predictions (RandomWalk + SARIMA)")
assert loaded == len(LAKES) * 2 * (60 + 58 + 55 + 49), "baseline window mismatch"


# VAR / VARX ROLLING under each future-climate policy

def build_exog(clim_frame):
    if clim_frame is None: return SEAS
    return pd.concat([SEAS, clim_frame], axis=1)

def var_rolling(label, clim_frame, future_policy):
    exog_full = build_exog(clim_frame)
    clim_cols = [] if clim_frame is None else list(clim_frame.columns)
    train0 = dL.iloc[1:split]; ex0 = exog_full.iloc[1:split]
    p = max(1, int(VAR(train0, exog=ex0).select_order(VAR_MAXLAG).aic))
    for o in targets:
        tr = dL.iloc[1:o]; ex_tr = exog_full.iloc[1:o]
        res = VAR(tr, exog=ex_tr).fit(p)
        H = min(max(HORIZONS), N - o)
        fut_idx = L.index[o:o + H]
        ex_future = exog_full.loc[fut_idx].copy()
        if clim_cols and future_policy != "oracle":
            if future_policy == "last":
                ex_future[clim_cols] = exog_full[clim_cols].iloc[o - 1].values
            elif future_policy == "zero":
                # OFF-BY-ONE CORRECTED
                precip_cols = [c for c in clim_cols if c.startswith("precip_")]
                if precip_cols:
                    ex_future.iloc[1:, [ex_future.columns.get_loc(c) for c in precip_cols]] = 0.0
                if "DMI" in clim_cols:
                    ex_future.iloc[1:, ex_future.columns.get_loc("DMI")] = exog_full["DMI"].iloc[o]
        fc_d = res.forecast(tr.values[-p:], steps=H, exog_future=ex_future.values)
        base = L.iloc[o - 1].values
        cum = base + np.cumsum(fc_d, axis=0)
        for h in HORIZONS:
            if h <= H:
                for j, lk in enumerate(LAKES):
                    preds[(label, lk, o + h - 1, h)] = cum[h - 1, j]
    print(f"  {label:14s}: VAR lag p={p}  (climate cols={len(clim_cols)}, policy={future_policy})")

CLIM_ALL = pd.concat([PRECIP_ALL, dmi.rename("DMI")], axis=1).shift(1).reindex(L.index).ffill().fillna(0.0)
CLIM_DMIONLY = pd.concat([dmi.rename("DMI")], axis=1).shift(1).reindex(L.index).ffill().fillna(0.0)

print("=== running variants ===")
var_rolling("VAR",         None,          "none")
var_rolling("VARX_orig",   CLIM_ALL,      "last")
var_rolling("VARX_zero",   CLIM_ALL,      "zero")
var_rolling("VARX_dmionly",CLIM_DMIONLY,  "zero")
var_rolling("VARX_oracle", CLIM_ALL,      "oracle")

# regression test: VARX_zero == VARX_oracle at h=1
_d = max(abs(preds[("VARX_zero", lk, t, 1)] - preds[("VARX_oracle", lk, t, 1)])
         for lk in LAKES for t in targets)
print(f"  identity check  max|VARX_zero - VARX_oracle| at h=1 = {_d:.2e}")
assert _d < 1e-9, "VARX_zero and VARX_oracle must coincide at h=1"


# SCORE (skill vs SARIMA)

MODELS = ["RandomWalk", "SARIMA", "VAR", "VARX_orig", "VARX_zero", "VARX_dmionly", "VARX_oracle"]
Lv = {lk: L[lk].values for lk in LAKES}
rows = []
for lk in LAKES:
    for model in MODELS:
        for h in HORIZONS:
            P, A = [], []
            for o in targets:
                tgt = o + h - 1
                if tgt >= N: continue
                key = (model, lk, tgt, h)
                if key in preds: P.append(preds[key]); A.append(Lv[lk][tgt])
            if P: rows.append({"Lake": lk, "Model": model, "Horizon_m": h,
                               "RMSE_m": round(rmse(P, A), 4),
                               "MAE_m": round(mae(P, A), 4)})     
m = pd.DataFrame(rows)
sar = m[m.Model == "SARIMA"].set_index(["Lake", "Horizon_m"])["RMSE_m"]
m["skill_vs_SARIMA_%"] = m.apply(lambda r: round(100 * (sar[(r.Lake, r.Horizon_m)] - r.RMSE_m) / sar[(r.Lake, r.Horizon_m)], 1), axis=1)
m.to_csv(os.path.join(DIAG_DIR, "VARX_diagnostic_metrics.csv"), index=False)

print("\n=== MEAN skill vs SARIMA (%) by model x horizon (positive = beats SARIMA) ===")
print(m.pivot_table(index="Model", columns="Horizon_m", values="skill_vs_SARIMA_%").reindex(MODELS).round(1).to_string())
print("\nSaved: VARX_diagnostic_metrics.csv")

  loaded 3108 shared-baseline predictions (RandomWalk + SARIMA)
=== running variants ===
  VAR           : VAR lag p=3  (climate cols=0, policy=none)
  VARX_orig     : VAR lag p=1  (climate cols=8, policy=last)
  VARX_zero     : VAR lag p=1  (climate cols=8, policy=zero)
  VARX_dmionly  : VAR lag p=3  (climate cols=1, policy=zero)
  VARX_oracle   : VAR lag p=1  (climate cols=8, policy=oracle)
  identity check  max|VARX_zero - VARX_oracle| at h=1 = 0.00e+00

=== MEAN skill vs SARIMA (%) by model x horizon (positive = beats SARIMA) ===
Horizon_m       1     3     6     12
Model                               
RandomWalk   -53.4 -65.8 -47.2  -0.8
SARIMA         0.0   0.0   0.0   0.0
VAR            0.5   2.7   3.3   0.3
VARX_orig     -8.9 -23.0 -45.9 -80.9
VARX_zero     11.7   6.0   4.1   3.0
VARX_dmionly   1.2   3.6   0.3 -13.0
VARX_oracle   11.7  18.2  15.6  13.9

Saved: VARX_diagnostic_metrics.csv
